# Mini Challenge — training & submission

Trains a simple **forecast-correction** model and produces a ready-to-upload
`submission.zip` (Codabench format). Use it as a baseline.

## 1. Setup

In [1]:
import os, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import lightgbm as lgb

_HERE = Path(os.getcwd()).resolve()
DATA = None
for _p in [_HERE, *_HERE.parents]:
    for cand in (_p / 'mini_challenge_dataset',
                 _p / 'participant_kit' / 'mini_challenge' / 'mini_challenge_dataset'):
        if cand.exists():
            DATA = cand; break
    if DATA: break
if DATA is None:
    raise FileNotFoundError('mini_challenge_dataset/ not found')
print('Dataset:', DATA)

meta = json.loads((DATA / 'buoy_metadata.json').read_text())
BUOYS = list(meta['buoys'].keys())
HOURS = meta['scoring_hours_utc']        # [0, 6, 12, 18]
print('Buoys:', BUOYS)

Dataset: /Users/david/projects/hackathon_2026/wind-prediction-private/participant_kit/mini_challenge/mini_challenge_dataset
Buoys: ['42001', '42036', '42059', '42060', '41043', '41044']


## 2. Build the training table (d1 / d7)

For every buoy we pair the HRES forecast (valid time, horizon d1/d7) with the observation at that valid time, resampled to the scoring hours.

In [2]:
def buoy_obs_6h(buoy):
    """Buoy obs resampled to 00/06/12/18 UTC."""
    o = pd.read_parquet(DATA / 'train' / 'buoy_observations' / f'ndbc_{buoy}.parquet')
    o['time'] = pd.to_datetime(o['time'])
    o = (o.set_index('time')[['wind_speed', 'wind_direction']]
          .resample('6h').mean().reset_index())
    return o[o['time'].dt.hour.isin(HOURS)]

rows = []
for buoy in BUOYS:
    hres = pd.read_parquet(DATA / 'train' / 'hres_forecasts' / f'hres_{buoy}.parquet')
    hres['time'] = pd.to_datetime(hres['time'])
    m = hres.merge(buoy_obs_6h(buoy), on='time', how='inner')
    m['station'] = buoy
    m['lat'] = meta['buoys'][buoy]['latitude']
    m['lon'] = meta['buoys'][buoy]['longitude']
    rows.append(m)
train = pd.concat(rows, ignore_index=True).dropna(
    subset=['wind_speed', 'hres_speed'])
print(f'Training rows: {len(train):,}  (horizons {sorted(train["horizon"].unique())})')

FEATURES = ['hres_speed', 'hres_u10', 'hres_v10', 'horizon', 'hour', 'lat', 'lon']
train['hres_dir_sin'] = np.sin(np.radians(train['hres_dir']))
train['hres_dir_cos'] = np.cos(np.radians(train['hres_dir']))
FEATURES += ['hres_dir_sin', 'hres_dir_cos']
print('Features:', FEATURES)

Training rows: 28,958  (horizons [np.int64(1), np.int64(7)])
Features: ['hres_speed', 'hres_u10', 'hres_v10', 'horizon', 'hour', 'lat', 'lon', 'hres_dir_sin', 'hres_dir_cos']


## 3. Train quantile models for wind speed

Three LightGBM models with the quantile objective → q05 / q50 / q95.
We hold out 2021 as a quick validation check.

In [3]:
tr = train[train['time'].dt.year == 2020]
va = train[train['time'].dt.year == 2021]
X_tr, y_tr = tr[FEATURES], tr['wind_speed']
X_va, y_va = va[FEATURES], va['wind_speed']

QUANTILES = {'q05': 0.05, 'q50': 0.50, 'q95': 0.95}
speed_models = {}
for name, q in QUANTILES.items():
    mdl = lgb.LGBMRegressor(objective='quantile', alpha=q, n_estimators=300,
                            learning_rate=0.05, num_leaves=31, min_child_samples=40,
                            verbose=-1)
    mdl.fit(X_tr, y_tr)
    speed_models[name] = mdl

# quick validation: Winkler + coverage on 2021
def winkler(actual, lo, hi, alpha=0.1):
    w = hi - lo
    w = w + np.where(actual < lo, 2/alpha*(lo-actual), 0)
    w = w + np.where(actual > hi, 2/alpha*(actual-hi), 0)
    return w
q05v = speed_models['q05'].predict(X_va)
q95v = speed_models['q95'].predict(X_va)
cov = np.mean((y_va.values >= q05v) & (y_va.values <= q95v))
print(f'2021 validation — Winkler: {winkler(y_va.values, q05v, q95v).mean():.3f}  '
      f'coverage: {cov:.1%} (target 90%)')
print('Top features:',
      sorted(zip(FEATURES, speed_models["q50"].feature_importances_),
             key=lambda t: -t[1])[:4])

2021 validation — Winkler: 6.039  coverage: 86.3% (target 90%)
Top features: [('hres_speed', np.int32(1657)), ('hres_v10', np.int32(1441)), ('hres_u10', np.int32(1415)), ('hres_dir_sin', np.int32(1201))]


## 4. Prediction helpers

- **d1 / d7** — apply the trained models to the window's HRES forecast.
- **d14** — climatology: empirical quantiles + circular-mean direction from
  the 14-day buoy context of that window.

In [4]:
def circular_mean(deg):
    deg = np.asarray(deg, float); deg = deg[np.isfinite(deg)]
    if len(deg) == 0:
        return 0.0
    a = np.radians(deg)
    return float(np.degrees(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) % 360)

def predict_hres(fc):
    """d1/d7 prediction from a window's HRES forecast frame."""
    fc = fc.copy()
    fc['hres_dir_sin'] = np.sin(np.radians(fc['hres_dir']))
    fc['hres_dir_cos'] = np.cos(np.radians(fc['hres_dir']))
    X = fc[FEATURES]
    out = {name: m.predict(X) for name, m in speed_models.items()}
    # enforce monotone quantiles
    q05 = np.minimum(out['q05'], out['q50'])
    q95 = np.maximum(out['q95'], out['q50'])
    return np.clip(q05, 0, None), out['q50'], q95, fc['hres_dir'].values

def predict_climatology(context_obs):
    """d14 fallback: empirical quantiles + circular-mean dir from the context."""
    ws = context_obs['wind_speed'].dropna()
    if len(ws) == 0:
        return 0.0, 5.0, 15.0, 0.0
    q05, q50, q95 = np.percentile(ws, [5, 50, 95])
    return q05, q50, q95, circular_mean(context_obs['wind_direction'])

## 5. Generate the submission

In [5]:
import zipfile

windows = sorted(int(p.name.split('_')[1])
                 for p in (DATA / 'inference').glob('window_*'))
out_rows = []
for w in windows:
    wdir = DATA / 'inference' / f'window_{w}'
    for buoy in BUOYS:
        b = meta['buoys'][buoy]
        ctx = pd.read_parquet(wdir / f'context_buoy_{buoy}.parquet')
        # d1 / d7 — from HRES forecast
        fc_path = wdir / f'forecast_hres_{buoy}.parquet'
        if fc_path.exists():
            fc = pd.read_parquet(fc_path)
            fc['lat'], fc['lon'] = b['latitude'], b['longitude']
            q05, q50, q95, d50 = predict_hres(fc)
            for i, r in fc.reset_index(drop=True).iterrows():
                out_rows.append(dict(window=w, station=buoy,
                    latitude=b['latitude'], longitude=b['longitude'],
                    horizon=int(r['horizon']), hour=int(r['hour']),
                    q05=q05[i], q50=q50[i], q95=q95[i], dir_50=d50[i]))
        # d14 — climatology fallback
        c05, c50, c95, cdir = predict_climatology(ctx)
        for hour in HOURS:
            out_rows.append(dict(window=w, station=buoy,
                latitude=b['latitude'], longitude=b['longitude'],
                horizon=14, hour=hour,
                q05=c05, q50=c50, q95=c95, dir_50=cdir))

sub = pd.DataFrame(out_rows).sort_values(
    ['window', 'station', 'horizon', 'hour']).reset_index(drop=True)
sub[['q05', 'q50', 'q95']] = sub[['q05', 'q50', 'q95']].round(3)
sub['dir_50'] = sub['dir_50'].round(1)

# --- Codabench-ready package ---------------------------------------------
# Codabench expects a .zip whose ONLY file is named exactly `predictions.csv`,
# at the archive root (no sub-folder). We write the CSV then zip it.
sub.to_csv('predictions.csv', index=False)
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('predictions.csv', arcname='predictions.csv')

print(f'submission.zip ready  -  {len(sub)} rows  (predictions.csv inside)')
print('  -> upload submission.zip directly to Codabench competition #13566')
print(sub.head(6).to_string(index=False))

submission.zip ready  -  576 rows  (predictions.csv inside)
  -> upload submission.zip directly to Codabench competition #13566
 window station  latitude  longitude  horizon  hour   q05   q50   q95  dir_50
      1   41043      21.1      -64.8        1     0 2.656 4.520 5.730    57.5
      1   41043      21.1      -64.8        1     6 3.457 4.972 6.444    79.3
      1   41043      21.1      -64.8        1    12 3.859 5.127 6.922    88.1
      1   41043      21.1      -64.8        1    18 2.440 3.580 5.556   106.2
      1   41043      21.1      -64.8        7     0 5.227 7.150 8.644   107.2
      1   41043      21.1      -64.8        7     6 3.602 6.061 8.271   115.4
